In [ ]:
import torch
import numpy as np
from msrtv import SE2, SIM2
from msrtv.utils import PSNR
from bm3d import bm3d
from tqdm.auto import tqdm
from PIL import Image
from pathlib import Path
import json
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 10.0,
    "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}",
    "image.cmap": "gray",
    "figure.constrained_layout.use": True,
    "xtick.top": False,
    "xtick.bottom": False,
    "ytick.left": False,
    "ytick.right": False,
    "xtick.labeltop": False,
    "xtick.labelbottom": False,
    "ytick.labelleft": False,
    "ytick.labelright": False,
})
latex_line_width = 4.804

device = "cuda"

In [ ]:
# colours
red = "#B6321C" # BrickRed
blue = "#0071BC" # RoyalBlue
green = "#009B55" # ForestGreen
purple = "#613F99" # RoyalPurple

def plot_image(ax, image, **kwargs):
    image = image.cpu()
    return ax.imshow(image, **kwargs)

output_directory = Path("output")
data_directory = Path("data")
config_path = Path("config.json")

In [ ]:
# "monalisa" "collagen" "spiral"
experiment = "collagen"

with open(config_path, encoding="utf-8") as f:
    config = json.load(f)[experiment]

def load_image(path):
    """Load image at `path` as a float32 tensor."""
    return torch.as_tensor(np.array(Image.open(path)), dtype=torch.float32, device=device)

ground_truth = load_image(data_directory / config["ground_truth"])
noisy = (load_image(data_directory / config["noisy"]))

In [ ]:
fig, ax = plt.subplots(1, 2)
plot_image(ax[0], ground_truth)
plot_image(ax[1], noisy);

In [ ]:
S, Or = 4, 32
H, W = ground_truth.shape
N_spatial = min(H, W)
λ = config["lambda"]

config_SE2 = config["SE2"]
ξ_SE2 = config_SE2["xi"]
σ_s_SE2 = config_SE2["σ_s"]
σ_o_SE2 = config_SE2["σ_o"]
T_SE2 = config_SE2["T"]

config_SIM2 = config["SIM2"]
ξ_SIM2 = config_SIM2["xi"]
φ = config_SIM2["phi"]
a = config_SIM2["a"]
σ_s_SIM2 = config_SIM2["σ_s"]
σ_o_SIM2 = config_SIM2["σ_o"]
σ_a = config_SIM2["σ_a"]
cut_freq = config_SIM2["cut_freq"]
T_SIM2 = config_SIM2["T"]

G_inv_SIM2 = (ξ_SIM2**-2, ξ_SIM2**-2, 1., φ**2)
G_inv_SE2 = (ξ_SE2**-2, ξ_SE2**-2, 1.)

quantiles_threshold = torch.linspace(0, 1, 5, device=device)

scales_bm3d = torch.arange(0., config["bm3d_max_scale"], config["bm3d_step_scale"])

In [ ]:
cakewavelets = SE2.cakewavelettransform.cakewavelet_stack_fourier(N_spatial, Or)
U_SE2 = SE2.cakewavelettransform.wavelet_transform(noisy, cakewavelets)
simoncelli_filters, lowpass = SIM2.simoncellitransform.steerable_wavelet_stack(H, W, S, Or, cut_freq=cut_freq, dilation=a)
U_SIM2, U_low = SIM2.simoncellitransform.wavelet_transform(noisy, simoncelli_filters, lowpass)

In [ ]:
thresholds = torch.quantile(
    U_SIM2.abs().flatten(start_dim=-2),
    q=torch.linspace(0, 1, 5, device=device),
    dim=-1,
    keepdim=True,
).unsqueeze(-1)
out = torch.nn.functional.relu(U_SIM2 - thresholds) - torch.nn.functional.relu(-U_SIM2 - thresholds)
out.shape

In [ ]:
ground_truth_SIM2 = SIM2.tvflow.GroundTruth(
    ground_truth=ground_truth,
    u_low=U_low,
    filters=simoncelli_filters,
    lowpass=lowpass,
)

In [ ]:
s = 1
o = 4

fig, ax = plt.subplots(2, 3, figsize=(24, 10))
cbar = plot_image(ax[0, 0], ground_truth)
fig.colorbar(cbar, ax=ax[0, 0])
cbar = plot_image(ax[1, 0], noisy)
fig.colorbar(cbar, ax=ax[1, 0])

cbar = plot_image(ax[0, 1], U_SE2[o])
fig.colorbar(cbar, ax=ax[0, 1])
cbar = plot_image(ax[1, 1], U_SE2.sum(-3) - noisy)
fig.colorbar(cbar, ax=ax[1, 1])

cbar = plot_image(ax[0, 2], U_SIM2[s, o])
fig.colorbar(cbar, ax=ax[0, 2])
cbar = plot_image(ax[1, 2], SIM2.simoncellitransform.inverse_wavelet_transform(
    U_SIM2, U_low, simoncelli_filters, lowpass
) - noisy)
fig.colorbar(cbar, ax=ax[1, 2]);

In [ ]:
def denoise_bm3d(u, ground_truth, scales, max_val=255.):
    PSNRs = []
    for σ in tqdm(scales):
        denoised = bm3d(u, σ)
        PSNRs.append(PSNR(torch.as_tensor(denoised), ground_truth, max_val=max_val))
    return denoised, PSNRs

In [ ]:
U_denoised_SE2, PSNRs_SE2 = SE2.tvflow.denoise(U_SE2, T_SE2, G_inv_SE2, σ_s_SE2, σ_o_SE2, ground_truth)
U_denoised_SIM2, PSNRs_SIM2 = SIM2.tvflow.denoise(U_SIM2, T_SIM2, G_inv_SIM2, a, σ_s_SIM2, σ_o_SIM2, σ_a, ground_truth_SIM2)
U_denoised_threshold, PSNRs_threshold = SIM2.threshold.denoise(U_SIM2, quantiles_threshold, ground_truth_SIM2)
denoised_bm3d, PSNRs_bm3d = denoise_bm3d(noisy.cpu(), ground_truth.cpu(), scales_bm3d)

In [ ]:
i_opt_SE2 = torch.tensor(PSNRs_SE2).argmax()
slice_SE2 = PSNRs_SE2[:2*i_opt_SE2]
i_opt_SIM2 = torch.tensor(PSNRs_SIM2).argmax()
slice_SIM2 = PSNRs_SIM2[:2*i_opt_SIM2]
i_opt_threshold = torch.tensor(PSNRs_threshold).argmax()
slice_threshold = PSNRs_threshold[:2*i_opt_threshold]
i_opt_bm3d = torch.tensor(PSNRs_bm3d).argmax()
slice_bm3d = PSNRs_bm3d[:2*i_opt_bm3d]

fig, ax = plt.subplots(1, 1)
ax.plot(torch.linspace(0, 2, len(slice_SIM2)), slice_SIM2, label="SIM(2)", color=red)
ax.plot(torch.linspace(0, 2, len(slice_SE2)), slice_SE2, label="SE(2)", color=green)
ax.plot(torch.linspace(0, 2, len(slice_threshold)), slice_threshold, label="Thresholding", color=blue)
ax.plot(torch.linspace(0, 2, len(slice_bm3d)), slice_bm3d, label="BM3D", color=purple)
ax.legend();

In [ ]:
Ts_SE2 = torch.linspace(0, T_SE2, len(PSNRs_SE2))
Ts_SIM2 = torch.linspace(0, T_SIM2, len(PSNRs_SIM2))
np.savetxt(output_directory / f"{experiment}_TV_SE2.dat", np.stack((Ts_SE2, PSNRs_SE2), axis=-1))
np.savetxt(output_directory / f"{experiment}_TV_SIM2.dat", np.stack((Ts_SIM2, PSNRs_SIM2), axis=-1))
np.savetxt(output_directory / f"{experiment}_threshold.dat", np.stack((quantiles_threshold.cpu(), PSNRs_threshold), axis=-1))
np.savetxt(output_directory / f"{experiment}_BM3D.dat", np.stack((scales_bm3d, PSNRs_bm3d), axis=-1))